In [ ]:
storage_account = "stretailbidevgk01"
container = "retail-data"

storage_key = dbutils.secrets.get(
    scope="kv-retail-bi-dev",
    key="adls-storage-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("ADLS authentication configured successfully")

In [ ]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    current_timestamp,
    lit
)

BASE_PATH = "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net"

GOLD_PATH = f"{BASE_PATH}/gold"
SILVER_PATH = f"{BASE_PATH}/silver"

Load the Gold tables

In [ ]:
df_dim_date = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_date/"
)

df_dim_customer = spark.read.format("delta").load(
    f"{GOLD_PATH}/dim_customer/"
)

df_fact_sales = spark.read.format("delta").load(
    f"{GOLD_PATH}/fact_sales/"
)

df_agg_daily_sales = spark.read.format("delta").load(
    f"{GOLD_PATH}/agg_daily_sales/"
)

print("DimDate:", df_dim_date.count())
print("DimCustomer:", df_dim_customer.count())
print("FactSales:", df_fact_sales.count())
print("AggDailySales:", df_agg_daily_sales.count())

FactSales DQ

In [ ]:
fact_sales_checks = {
    "Duplicate OrderLineID": (
        df_fact_sales
        .groupBy("OrderLineID")
        .count()
        .filter(col("count") > 1)
        .count()
    ),
    "Null CustomerID": df_fact_sales.filter(
        col("CustomerID").isNull()
    ).count(),
    "Null OrderDateKey": df_fact_sales.filter(
        col("OrderDateKey").isNull()
    ).count(),
    "Invalid Quantity": df_fact_sales.filter(
        col("Quantity") <= 0
    ).count(),
    "Invalid UnitPrice": df_fact_sales.filter(
        col("UnitPrice") < 0
    ).count()
}

for check, result in fact_sales_checks.items():
    print(f"{check}: {result}")

DimCustomer DQ

In [ ]:
customer_checks = {
    "Null CustomerID": df_dim_customer.filter(
        col("CustomerID").isNull()
    ).count(),
    "Duplicate CustomerID": (
        df_dim_customer
        .groupBy("CustomerID")
        .count()
        .filter(col("count") > 1)
        .count()
    )
}

for check, result in customer_checks.items():
    print(f"{check}: {result}")

DimDate DQ

In [ ]:
date_checks = {
    "Null Date": df_dim_date.filter(
        col("Date").isNull()
    ).count(),
    "Null DateKey": df_dim_date.filter(
        col("DateKey").isNull()
    ).count(),
    "Duplicate DateKey": (
        df_dim_date
        .groupBy("DateKey")
        .count()
        .filter(col("count") > 1)
        .count()
    )
}

for check, result in date_checks.items():
    print(f"{check}: {result}")

FactSales → DimCustomer

In [ ]:
customer_fk_violations = (
    df_fact_sales
    .select("CustomerID")
    .distinct()
    .join(
        df_dim_customer.select("CustomerID").distinct(),
        on="CustomerID",
        how="left_anti"
    )
    .count()
)

print("FactSales → DimCustomer FK violations:", customer_fk_violations)

FactSales → DimDate

In [ ]:
date_fk_violations = (
    df_fact_sales
    .select("OrderDateKey")
    .distinct()
    .join(
        df_dim_date.select("DateKey").distinct(),
        df_fact_sales["OrderDateKey"] == df_dim_date["DateKey"],
        how="left_anti"
    )
    .count()
)

print("FactSales → DimDate FK violations:", date_fk_violations)

Create DQ results

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)
from datetime import datetime
from pyspark.sql.functions import when

dq_results = [
    ("dim_date", "Null Date", "Completeness", 0,
     df_dim_date.filter(col("Date").isNull()).count()),

    ("dim_date", "Null DateKey", "Completeness", 0,
     df_dim_date.filter(col("DateKey").isNull()).count()),

    ("dim_date", "Duplicate DateKey", "Uniqueness", 0,
     df_dim_date.groupBy("DateKey").count()
     .filter(col("count") > 1).count()),

    ("dim_customer", "Null CustomerID", "Completeness", 0,
     df_dim_customer.filter(col("CustomerID").isNull()).count()),

    ("dim_customer", "Duplicate CustomerID", "Uniqueness", 0,
     df_dim_customer.groupBy("CustomerID").count()
     .filter(col("count") > 1).count()),

    ("fact_sales", "Duplicate OrderLineID", "Uniqueness", 0,
     df_fact_sales.groupBy("OrderLineID").count()
     .filter(col("count") > 1).count()),

    ("fact_sales", "Null CustomerID", "Completeness", 0,
     df_fact_sales.filter(col("CustomerID").isNull()).count()),

    ("fact_sales", "Null OrderDateKey", "Completeness", 0,
     df_fact_sales.filter(col("OrderDateKey").isNull()).count()),

    ("fact_sales", "Invalid Quantity", "Validity", 0,
     df_fact_sales.filter(col("Quantity") <= 0).count()),

    ("fact_sales", "Invalid UnitPrice", "Validity", 0,
     df_fact_sales.filter(col("UnitPrice") < 0).count()),

    ("fact_sales", "Customer FK violations", "Referential Integrity", 0,
     customer_fk_violations),

    ("fact_sales", "Date FK violations", "Referential Integrity", 0,
     date_fk_violations)
]

schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("check_type", StringType(), False),
    StructField("expected_value", IntegerType(), False),
    StructField("actual_value", IntegerType(), False)
])

df_dq_results = spark.createDataFrame(
    dq_results,
    schema=schema
).withColumn(
    "status",
    when(col("actual_value") == col("expected_value"), "PASS")
    .otherwise("FAIL")
).withColumn(
    "check_timestamp",
    current_timestamp()
)

display(df_dq_results)

Verify overall status

In [ ]:
display(
    df_dq_results
    .groupBy("status")
    .count()
)

Write DQ results to Gold

In [ ]:
dq_results_path = (
    "abfss://retail-data@stretailbidevgk01.dfs.core.windows.net/"
    "gold/dq_results/"
)

(
    df_dq_results
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_results_path)
)

Verify

In [ ]:
df_dq_check = (
    spark.read
    .format("delta")
    .load(dq_results_path)
)

print("DQ result rows:", df_dq_check.count())

display(
    df_dq_check.orderBy("table_name", "check_name")
)